# DeepCFR Network Unit Tests

**Comprehensive tests for the DeepCFR neural network module**

Tests the network with real parsed infosets from MCCFR.

In [ ]:
import sys
import os

# Add parent directory to path
current_dir = os.getcwd()
if current_dir.endswith('deep_CFR_vNB_integration'):
    parent_dir = os.path.dirname(current_dir)
else:
    parent_dir = os.path.dirname(os.path.dirname(current_dir))
sys.path.insert(0, parent_dir)

import torch
import random
import numpy as np
from DeepCFR import DeepCFRModule
from infoset_parser import parse_infoset_to_network_input, batch_parse_infosets
from mccfr import MCCFR

# Test tracking
tests_passed = 0
tests_failed = 0

def run_test(test_name, test_func):
    global tests_passed, tests_failed
    try:
        test_func()
        print(f'✓ {test_name} PASSED')
        tests_passed += 1
    except AssertionError as e:
        print(f'✗ {test_name} FAILED: {e}')
        tests_failed += 1
    except Exception as e:
        print(f'✗ {test_name} ERROR: {e}')
        tests_failed += 1

print('=' * 70)
print('DEEPCFR NETWORK UNIT TESTS')
print('=' * 70)

## 1. Network Initialization Tests

In [ ]:
def test_network_creates_successfully():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    assert isinstance(network, torch.nn.Module)
    assert network.n_action_history == 20

run_test('Network creates successfully', test_network_creates_successfully)

In [ ]:
def test_network_parameter_count():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    params = sum(p.numel() for p in network.parameters() if p.requires_grad)
    assert params > 0
    print(f'  Network has {params:,} trainable parameters')

run_test('Network has trainable parameters', test_network_parameter_count)

In [ ]:
def test_network_embedding_layers():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    assert len(network.hand_embeddings) == 3
    assert len(network.board_embeddings) == 5

run_test('Network has correct embeddings', test_network_embedding_layers)

## 2. Forward Pass - Single Sample Tests

In [ ]:
def test_forward_pass_preflop():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S0|H:14s0,13s1,10s2|B:|A:"
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        output = network(cc, ah)
    assert output.shape == torch.Size([1, 9])
    assert output.dtype == torch.float32

run_test('Forward pass: Preflop', test_forward_pass_preflop)

In [ ]:
def test_forward_pass_with_board():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S1|H:14s0,13s0,10s1|B:9s0,8s1,7s2,6s0,5s1|A:CRDD"
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        output = network(cc, ah)
    assert output.shape == torch.Size([1, 9])
    assert len(cc.canonical_board) == 5

run_test('Forward pass: With board', test_forward_pass_with_board)

In [ ]:
def test_forward_pass_with_action_history():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S2|H:14s0,13s0,10s1|B:9s0,8s1,7s2,6s0,5s1|A:CRBXrDFC"
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        output = network(cc, ah)
    assert output.shape == torch.Size([1, 9])
    assert ah == "CRBXrDFC"

run_test('Forward pass: With action history', test_forward_pass_with_action_history)

## 3. Forward Pass - Batch Tests

In [ ]:
def test_batch_forward_pass():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infosets = [
        "S0|H:14s0,13s1,10s2|B:|A:",
        "S0|H:12s0,11s1,9s2|B:|A:C",
        "S1|H:8s0,7s1,6s2|B:5s0,4s1,3s2,2s0,14s1|A:DD",
    ]
    cc_list, ah_list = batch_parse_infosets(infosets)
    with torch.no_grad():
        output = network(cc_list, ah_list)
    assert output.shape == torch.Size([3, 9])
    assert output.dtype == torch.float32

run_test('Batch forward pass: 3 samples', test_batch_forward_pass)

In [ ]:
def test_batch_size_10():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infosets = []
    for i in range(10):
        rank1, rank2, rank3 = 14-i, 13-i, 10
        infosets.append(f"S0|H:{rank1}s0,{rank2}s1,{rank3}s2|B:|A:")
    cc_list, ah_list = batch_parse_infosets(infosets)
    with torch.no_grad():
        output = network(cc_list, ah_list)
    assert output.shape == torch.Size([10, 9])

run_test('Batch forward pass: 10 samples', test_batch_size_10)

## 4. Real MCCFR Infosets Tests

In [ ]:
def test_with_real_mccfr_infosets():
    random.seed(42)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    mccfr = MCCFR()
    infosets = []
    for i in range(5):
        state = mccfr.create_initial_state()
        infoset = mccfr.get_infoset(state, 0)
        infosets.append(infoset)
    cc_list, ah_list = batch_parse_infosets(infosets)
    with torch.no_grad():
        output = network(cc_list, ah_list)
    assert output.shape == torch.Size([5, 9])
    print(f'  Successfully processed {len(infosets)} real MCCFR infosets')

run_test('Network with real MCCFR infosets', test_with_real_mccfr_infosets)

In [ ]:
def test_with_mccfr_game_progression():
    random.seed(123)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    mccfr = MCCFR()
    state = mccfr.create_initial_state()
    infoset_preflop = mccfr.get_infoset(state, 0)
    cc, ah = parse_infoset_to_network_input(infoset_preflop)
    with torch.no_grad():
        output = network(cc, ah)
    assert output.shape == torch.Size([1, 9])
    assert state.street == 0
    print(f'  Tested infoset from street {state.street}')

run_test('Network with game progression', test_with_mccfr_game_progression)

## 5. Action History Encoding Tests

In [ ]:
def test_action_history_empty():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    encoded = network._encode_action_history("")
    assert encoded.shape == torch.Size([120])
    assert torch.all(encoded == 0.0)

run_test('Action history: Empty', test_action_history_empty)

In [ ]:
def test_action_history_single_action():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    test_cases = [('X', 0), ('C', 1), ('F', 2), ('D', 3), ('r', 4), ('R', 5), ('B', 5)]
    for action_char, expected_idx in test_cases:
        encoded = network._encode_action_history(action_char)
        first_action = encoded[:6]
        expected = torch.zeros(6)
        expected[expected_idx] = 1.0
        assert torch.allclose(first_action, expected)

run_test('Action history: Single actions', test_action_history_single_action)

In [ ]:
def test_action_history_long_sequence():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    long_history = "CRBXDFCRBXDFCRBXDFCRBXDF"
    encoded = network._encode_action_history(long_history)
    assert encoded.shape == torch.Size([120])
    first_action = encoded[:6]
    expected = torch.zeros(6)
    expected[1] = 1.0  # C = call
    assert torch.allclose(first_action, expected)

run_test('Action history: Long sequence', test_action_history_long_sequence)

## 6. Edge Case Tests

In [ ]:
def test_empty_board_preflop():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S0|H:14s0,13s1,10s2|B:|A:"
    cc, ah = parse_infoset_to_network_input(infoset)
    assert len(cc.canonical_board) == 0
    with torch.no_grad():
        output = network(cc, ah)
    assert output.shape == torch.Size([1, 9])

run_test('Edge case: Empty board', test_empty_board_preflop)

In [ ]:
def test_all_same_suit():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S0|H:14s0,13s0,10s0|B:|A:"
    cc, ah = parse_infoset_to_network_input(infoset)
    suits = [card.split('s')[1] for card in cc.canonical_hand]
    assert len(set(suits)) == 1
    with torch.no_grad():
        output = network(cc, ah)
    assert output.shape == torch.Size([1, 9])

run_test('Edge case: All same suit', test_all_same_suit)

In [ ]:
def test_output_values_are_finite():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S1|H:14s0,13s0,10s1|B:9s0,8s1,7s2,6s0,5s1|A:CRDD"
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        output = network(cc, ah)
    assert torch.all(torch.isfinite(output))
    assert not torch.any(torch.isnan(output))
    assert not torch.any(torch.isinf(output))

run_test('Edge case: Output values finite', test_output_values_are_finite)

## 7. Network Consistency Tests

In [ ]:
def test_deterministic_output():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset = "S0|H:14s0,13s1,10s2|B:|A:"
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        output1 = network(cc, ah)
        output2 = network(cc, ah)
    assert torch.allclose(output1, output2)

run_test('Consistency: Deterministic output', test_deterministic_output)

In [ ]:
def test_different_inputs_different_outputs():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.eval()
    infoset1 = "S0|H:14s0,13s1,10s2|B:|A:"
    infoset2 = "S0|H:12s0,11s1,9s2|B:|A:"
    cc1, ah1 = parse_infoset_to_network_input(infoset1)
    cc2, ah2 = parse_infoset_to_network_input(infoset2)
    with torch.no_grad():
        output1 = network(cc1, ah1)
        output2 = network(cc2, ah2)
    assert not torch.allclose(output1, output2)

run_test('Consistency: Different inputs', test_different_inputs_different_outputs)

In [ ]:
def test_gradient_flow():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    network.train()
    infoset = "S0|H:14s0,13s1,10s2|B:|A:"
    cc, ah = parse_infoset_to_network_input(infoset)
    output = network(cc, ah)
    loss = output.sum()
    loss.backward()
    has_gradients = False
    for param in network.parameters():
        if param.grad is not None and param.grad.abs().sum() > 0:
            has_gradients = True
            break
    assert has_gradients

run_test('Consistency: Gradient flow', test_gradient_flow)

## Test Summary

In [ ]:
print('\n' + '=' * 70)
print('DEEPCFR NETWORK TEST SUMMARY')
print('=' * 70)
print(f'\nTests passed: {tests_passed}')
print(f'Tests failed: {tests_failed}')
print(f'Total tests: {tests_passed + tests_failed}')
if tests_failed == 0:
    print('\n✓✓✓ ALL TESTS PASSED! ✓✓✓')
    print('\nNetwork verified for:')
    print('  ✓ Single and batch forward passes')
    print('  ✓ Real MCCFR infosets')
    print('  ✓ Various game states')
    print('  ✓ Action history encoding')
    print('  ✓ Edge cases and consistency')
    print('  ✓ Gradient flow for training')
    print('\nReady to proceed to Step 5: Network-MCCFR Integration')
else:
    print(f'\n✗ {tests_failed} TEST(S) FAILED')
print('=' * 70)